In [1]:
import originpro as op
import numpy as np
import pandas as pd

import os
# Very useful, especially during development, when you are
# liable to have a few uncaught exceptions.
# Ensures that the Origin instance gets shut down properly.
# Note: only applicable to external Python.
import sys
def origin_shutdown_exception_hook(exctype, value, traceback):
    '''Ensures Origin gets shut down if an uncaught exception'''
    op.exit()
    sys.__excepthook__(exctype, value, traceback)
if op and op.oext:
    sys.excepthook = origin_shutdown_exception_hook


# Set Origin instance visibility.
# Important for only external Python.
# Should not be used with embedded Python.
if op.oext:
    op.set_show(True)


# Example of opening a project and reading data.

# We'll open the Tutorial Data.opju project that ships with Origin.
src_opju = r"C:\usrspace\mywork\edges2.opju"   # <— 用绝对路径；注意路径和文件名是否正确
print("File exists? ", os.path.exists(src_opju))
ok = op.open(file=src_opju)
print("Project opened? ", ok)


File exists?  True
Project opened?  True


In [2]:

# ===== 2) 列出所有页面类型，看看有没有工作簿 =====
wbooks = list(op.pages('w'))   # 所有工作簿（Worksheet Book）
mbks   = list(op.pages('m'))   # 所有矩阵簿（Matrix Book）
graphs = list(op.pages('g'))   # 所有图页（Graph），仅排查
notes  = list(op.pages('n'))   # 所有Notes，仅排查

print("#Workbooks:", len(wbooks), "| #MatrixBooks:", len(mbks), "| #Graphs:", len(graphs), "| #Notes:", len(notes))

# ===== 3) 如果有工作簿：打印每个工作簿下所有工作表名 =====
for wb in wbooks:
    sheet_names = [wks.name for wks in wb]
    print(f"[{wb.name}] -> {sheet_names}")

# ===== 4) 如果是矩阵簿：打印矩阵表名（有些项目只有矩阵簿，没有工作簿）=====
for mb in mbks:
    ms_names = [ms.name for ms in mb]
    print(f"[{mb.name}] (Matrix) -> {ms_names}")

# ===== 5) 若你想强行拿到第一个工作簿并遍历 =====
if wbooks:
    wb = wbooks[0]
    print("Active to:", wb.name)
    wb.activate()  # 让它成为激活窗口
    for wks in wb:
        print("Sheet:", wks.name)

#Workbooks: 1 | #MatrixBooks: 0 | #Graphs: 43 | #Notes: 0
[Book1] -> ['pending_edges_ttb30', 'pending_edges_ttb40', 'pending_edges_ttb50', 'pending_edges_ttb60', 'pending_edges_ttb70', 'pending_edges_ttb80', 'pending_edges_ttb90', 'pending_edges_ttb100', 'pending_edges_ttb110', 'pending_edges_ttb120', 'pending_edges_ttb130', 'pending_edges_ttb140', 'stable_duration_time', 'pending_edges_ttb10', 'pending_edges_ttb20', 'Ration']
Active to: Book1
Sheet: pending_edges_ttb30
Sheet: pending_edges_ttb40
Sheet: pending_edges_ttb50
Sheet: pending_edges_ttb60
Sheet: pending_edges_ttb70
Sheet: pending_edges_ttb80
Sheet: pending_edges_ttb90
Sheet: pending_edges_ttb100
Sheet: pending_edges_ttb110
Sheet: pending_edges_ttb120
Sheet: pending_edges_ttb130
Sheet: pending_edges_ttb140
Sheet: stable_duration_time
Sheet: pending_edges_ttb10
Sheet: pending_edges_ttb20
Sheet: Ration


In [3]:
# 下面是生成平均无建链时间和无建链总时间占比

import originpro as op
import pandas as pd
from pathlib import Path

# 可调参数
MIN_WINDOW_LEN = 1    # 只统计长度 >= 这个阈值的 0 窗口（不想过滤就设 1）

results = []

for wb in op.pages('w'):           # 遍历所有工作簿
    for wks in wb:                 # 遍历每个工作表
        nm = wks.name
        if not nm.startswith('pending_edges_'):
            continue

        # ===== 读取 =====
        df = wks.to_df(head='L')
        if not {'time','pending_edges'}.issubset(df.columns):
            print(f"跳过（缺列）: [{wb.name}] {nm}")
            continue

        # ===== 规范化 =====
        df = df[['time','pending_edges']].copy()
        try:
            df['time'] = df['time'].astype(int)
        except Exception:
            df['time'] = pd.to_numeric(df['time'], errors='coerce')
            df = df.dropna(subset=['time'])
            df['time'] = df['time'].astype(int)
        df['pending_edges'] = pd.to_numeric(df['pending_edges'], errors='coerce')

        # ===== 映射 & 连续性检查 =====
        mapping = df.set_index('time')['pending_edges'].to_dict()
        times = sorted(mapping.keys())
        if not times:
            print(f"跳过（空表）: [{wb.name}] {nm}")
            continue

        t_min, t_max = times[0], times[-1]
        gaps = []
        for a, b in zip(times, times[1:]):
            if b - a > 1:
                gaps.append((a+1, b-1))
        missing_total = sum((b - a + 1) for (a, b) in gaps) if gaps else 0

        # ===== 补缺口为 0（断了就算稳定）=====
        size = t_max - t_min + 1
        edges = [0] * size
        for t, e in mapping.items():
            idx = t - t_min
            if pd.notna(e):
                try:
                    edges[idx] = int(e)
                except Exception:
                    try:
                        edges[idx] = int(float(e))
                    except Exception:
                        edges[idx] = 0

        # ===== 找出所有 edge==0 的连续窗口 =====
        windows = []  # [(start_t, end_t, length_s), ...]
        cur_len = 0
        cur_start_idx = None

        for i, v in enumerate(edges):
            if v == 0:
                if cur_len == 0:
                    cur_start_idx = i
                cur_len += 1
            else:
                if cur_len >= MIN_WINDOW_LEN and cur_start_idx is not None:
                    s = t_min + cur_start_idx
                    L = cur_len
                    e = s + L - 1
                    windows.append((s, e, L))
                cur_len = 0
                cur_start_idx = None

        # 收尾
        if cur_len >= MIN_WINDOW_LEN and cur_start_idx is not None:
            s = t_min + cur_start_idx
            L = cur_len
            e = s + L - 1
            windows.append((s, e, L))

        # ===== 最长窗口 =====
        if windows:
            s_long, e_long, L_long = max(windows, key=lambda x: x[2])
            longest_len_min = L_long / 60.0
        else:
            s_long = e_long = None
            L_long = 0
            longest_len_min = 0.0

        # ===== 平均窗口时长 =====
        if windows:
            avg_s = sum(L for _,_,L in windows) / len(windows)
            avg_min = avg_s / 60.0
            n_windows = len(windows)
        else:
            avg_s = avg_min = 0.0
            n_windows = 0

        # ===== 新增：稳定时间与占比 =====
        stable_seconds = sum(L for _,_,L in windows) if windows else 0
        theoretical_seconds = (t_max - t_min + 1)               # 理论总秒数
        stable_ratio = (stable_seconds / theoretical_seconds) if theoretical_seconds > 0 else 0.0
        stable_ratio_pct = stable_ratio * 100.0

        print(f"[{wb.name}] {nm} | time[{t_min},{t_max}] "
              f"| gaps={len(gaps)} miss={missing_total}s "
              f"| longest={L_long}s ({longest_len_min:.2f}m) "
              f"| avg={avg_s:.2f}s ({avg_min:.2f}m) over {n_windows} windows "
              f"| stable={stable_seconds}s / {theoretical_seconds}s = {stable_ratio_pct:.2f}%")

        results.append({
            'book': wb.name, 'sheet': nm,
            'time_min': t_min, 'time_max': t_max,
            'observed_seconds': len(times),
            'theoretical_seconds': theoretical_seconds,
            'n_gaps': len(gaps), 'missing_total_seconds': missing_total,
            'longest_start': s_long, 'longest_end': e_long,
            'longest_len_s': L_long, 'longest_len_min': round(longest_len_min, 4),
            'avg_len_s': round(avg_s, 4), 'avg_len_min': round(avg_min, 4),
            'n_windows': n_windows, 'min_window_len_used': MIN_WINDOW_LEN,
            # 新增字段
            'stable_seconds': stable_seconds,
            'stable_ratio': round(stable_ratio, 6),
            'stable_ratio_pct': round(stable_ratio_pct, 4),
        })


[Book1] pending_edges_ttb30 | time[3,21893] | gaps=0 miss=0s | longest=158s (2.63m) | avg=112.27s (1.87m) over 139 windows | stable=15605s / 21891s = 71.29%
[Book1] pending_edges_ttb40 | time[0,21893] | gaps=0 miss=0s | longest=148s (2.47m) | avg=114.16s (1.90m) over 129 windows | stable=14726s / 21894s = 67.26%
[Book1] pending_edges_ttb50 | time[0,21893] | gaps=0 miss=0s | longest=138s (2.30m) | avg=101.86s (1.70m) over 131 windows | stable=13344s / 21894s = 60.95%
[Book1] pending_edges_ttb60 | time[0,21893] | gaps=0 miss=0s | longest=128s (2.13m) | avg=92.98s (1.55m) over 128 windows | stable=11902s / 21894s = 54.36%
[Book1] pending_edges_ttb70 | time[0,21893] | gaps=0 miss=0s | longest=118s (1.97m) | avg=84.91s (1.42m) over 125 windows | stable=10614s / 21894s = 48.48%
[Book1] pending_edges_ttb80 | time[0,21893] | gaps=0 miss=0s | longest=108s (1.80m) | avg=79.06s (1.32m) over 125 windows | stable=9882s / 21894s = 45.14%
[Book1] pending_edges_ttb90 | time[0,21893] | gaps=0 miss=0s |

In [ ]:
# ===== 汇总并写回 Origin(写到 Book1) + 导出 CSV =====

# 写入测试
# 1) 找到或创建 Book1
wb = op.find_book('w', 'Book1')
if wb is None:
    wb = op.new_book('w', lname='Book1')   # 没有就新建

# 2) 在 Book1 里新建空表，并激活到前台
try:
    wks = wb.add_sheet('MyEmptySheet', active=True)   # 某些版本支持 name + active
except TypeError:
    wks = wb.add_sheet()              # 兼容：如果不支持命名/active参数
    wks.name = 'MyEmptySheet'
    wks.activate()

wb.activate()                          # 激活工作簿窗口
print("Created:", wks.lt_range())

# 3)（可选）列出 Book1 里的所有表，确认新表存在
print("Book1 sheets:", [s.name for s in wb])

if results:
    df_sum = pd.DataFrame(results)

    def _pick_ttb(s):
        try:
            return int(''.join(ch for ch in s if ch.isdigit()))
        except Exception:
            return None
    df_sum['ttb'] = df_sum['sheet'].map(_pick_ttb)
    df_sum = df_sum.sort_values(['book','ttb'], na_position='last')

    cols = ['sheet','ttb','avg_duration', 'stable_duration_ratio']


    print("\n=== SUMMARY ===")
    print(df_sum[cols].to_string(index=False))

    # —— 写到当前项目里的 Book1 —— #
    bk = op.find_book('w', 'Book1')              # 注意：先传类型 'w' 再传名字

    # 找 summary 表；有则用，没有就创建


    ws = bk.add_sheet('Ration', active=True)



    # 把汇总写入 summary 表
    ws.from_df(df_sum[cols])
    print("写入完成 -> [Book1]summary")
else:
    print("没有匹配到 pending_edges_* 的工作表。")



接下来的内容是,我们要计算,平均建立链路条数


